# 1. Motivation

### Selecting the Dataset
We wanted a dataset that was both familiar and rich enough to tell an interesting story. Netflix is a platform most people use, and the data behind it turns out to be surprisingly deep. The core dataset `netflix_titles.csv`, covers **8,807 titles** from 127 countries that were added to the platform between 2008 and 2021. We enriched it with TMDB community ratings and popularity scores, as well as production budget data for a selection of TV series. Together, these sources can answer a range of questions about how Netflix grew, what it offers, and how audiences use it.

### Why Netflix?
Netflix started out as a movie rental service focused in the US and quietly became one of the world's most biggest content platforms. We wanted to see whether that transformation was visible in the data, and as it turns out, it very much is. The dataset is large enough to show real patterns and structured enough to support their analysis.

### Goal for the end user
Our goal is to show a clear picture of how Netflix evolved over time and what that says about the content we watch today. Every chart is meant to add an agle to the story

# 2. Basic stats. Let's understand the dataset better
- Write about your choices in data cleaning and preprocessing
- Write a short section that discusses the dataset stats, containing key points/plots from your exploratory data analysis.


## Data sources

Four raw files go into this analysis:

- **`netflix_titles.csv`**: 8,807 rows, one per title. The core Netflix catalog snapshot: title, type (Movie / TV Show), director, cast, country, date added, release year, content rating, duration, and genre tags (`listed_in`). Pulled from Kaggle, scraped from Netflix at a single point in time.
- **`movies_metadata.csv`**: TMDB metadata for 45,000+ movies, collected via the TMDB API. Used here for `vote_average`, `popularity`, `vote_count`, `budget`, `revenue`, and `poster_path`.
- **`tvs.csv`**: TMDB metadata for TV series. Same fields minus `budget`/`revenue`, used to fill in ratings and popularity for TV Show rows.
- **`expensive_tv_series_with_budget.csv`**: per-season production budgets for a curated set of well-known series. Aggregated to series level and merged last to fill the `budget` column where data existed.

In [1]:
import pandas as pd

data_folder = './data/raw/'
titles_df = pd.read_csv(data_folder + 'netflix_titles.csv')
movie_budgets_df = pd.read_csv(data_folder + 'movies_metadata.csv')
series_budgets_df = pd.read_csv(data_folder + 'expensive_tv_series_with_budget.csv')
tv_ratings_df = pd.read_csv(data_folder + 'tvs.csv')

FileNotFoundError: [Errno 2] No such file or directory: './data/raw/netflix_titles.csv'

## Merging movie metadata

`movies_metadata.csv` is joined to the Netflix catalog on exact title match. Before merging, duplicate titles in the TMDB source are dropped (keeping the first occurrence). Only seven columns are kept from the TMDB side: `budget`, `popularity`, `poster_path`, `revenue`, `vote_average`, `vote_count`, and `title_movie`.

A left join preserves all 8,807 Netflix rows; unmatched titles get NaN for the TMDB fields. Exact-match joining is conservative: it misses titles with punctuation differences or subtitle variations, but it avoids the false matches a fuzzy strategy would introduce. Movie coverage comes out at 2,301 out of 6,131 (37.5%).

In [4]:
movies_count = (titles_df['type'] == 'Movie').sum()
tv_shows_count = (titles_df['type'] == 'TV Show').sum()

# Prepare movie_budgets_df: drop duplicates
movie_dups = movie_budgets_df['title'].duplicated().sum()
movie_budgets_clean = movie_budgets_df.drop_duplicates(subset=['title'], keep='first')
movie_budgets_clean.rename(columns={'title': 'title_movie'}, inplace=True)

keep_cols = ['budget', 'popularity', 'poster_path', 'revenue', 'vote_average', 'vote_count', 'title_movie']
movie_budgets_clean = movie_budgets_clean[keep_cols]

consolidated_df = titles_df.merge(
    movie_budgets_clean,
    left_on='title',
    right_on='title_movie',
    how='left',
)


exact_movie_matches = (
    (consolidated_df['type'] == 'Movie') & 
    (consolidated_df['title_movie'].notna())
).sum()
print(f"Exact movie matches: {exact_movie_matches} out of {movies_count} movies")

Exact movie matches: 2117 out of 6131 movies


## Merging TV show metadata

The same approach applies for `tvs.csv`. The source is de-duplicated on `name`, five columns are kept (`name`, `poster_path`, `vote_average`, `vote_count`, `popularity`), and a left join on title fills in TMDB data for TV shows.

TV shows match at a much higher rate: 2,155 out of 2,676 (80.5%), compared to 37.5% for movies. That gap probably reflects TMDB's origins as a general movie database. Its TV catalog grew more recently and tends to have better coverage for titles on major streaming platforms.

After both TMDB merges, the `_x`/`_y` column pairs from the two separate joins are collapsed into single columns using `combine_first`, keeping whichever side has a non-null value.

In [5]:
# Prepare tv_ratings_df: drop duplicates
tv_dups = tv_ratings_df['name'].duplicated().sum()
print(f"  - Duplicates found: {tv_dups}")

tv_ratings_clean = tv_ratings_df.drop_duplicates(subset=['name'], keep='first')
print(f"  - After drop_duplicates: {len(tv_ratings_clean)} rows")

tv_keep_cols = ['name', 'poster_path', 'vote_average', 'vote_count', 'popularity']

tv_ratings_clean = tv_ratings_clean[tv_keep_cols]

# Step 1: Exact match merge for TV shows
print(f"  - Before merge: {len(consolidated_df)} rows")

consolidated_df = consolidated_df.merge(
    tv_ratings_clean,
    left_on='title',
    right_on='name',
    how='left',
)

print(f"  - After merge: {len(consolidated_df)} rows")

exact_tv_matches = (
    (consolidated_df['type'] == 'TV Show') & 
    (consolidated_df['name'].notna())
).sum()
print(f"  - Exact TV matches: {exact_tv_matches} out of {tv_shows_count} TV shows")

  - Duplicates found: 8513
  - After drop_duplicates: 144457 rows
  - Before merge: 8807 rows
  - After merge: 8807 rows
  - Exact TV matches: 2137 out of 2676 TV shows


In [6]:
consolidated_df['popularity'] = consolidated_df['popularity_x'].combine_first(consolidated_df['popularity_y'])
consolidated_df['vote_average'] = consolidated_df['vote_average_x'].combine_first(consolidated_df['vote_average_y'])
consolidated_df['vote_count'] = consolidated_df['vote_count_x'].combine_first(consolidated_df['vote_count_y'])
consolidated_df['poster_path'] = consolidated_df['poster_path_x'].combine_first(consolidated_df['poster_path_y'])
consolidated_df.drop(columns=[
    'popularity_x', 'popularity_y', 
    'vote_average_x', 'vote_average_y', 
    'vote_count_x', 'vote_count_y', 
    'poster_path_x', 'poster_path_y',
    'name'
], inplace=True)
consolidated_df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,budget,revenue,title_movie,popularity,vote_average,vote_count,poster_path
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",NaN,NaN,NaN,14.193,7.900,285.0,/NFHy3z1taQ8nzudDHeVIJnih3J.jpg
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,NaN,NaN,NaN,30.808,7.090,133.0,/3E6IPkHH541ii41NKpU5loI4fcr.jpg
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",NaN,NaN,NaN,2.527,6.100,5.0,/pA7urHEBVDSJR7Cn7bANRbhwly7.jpg
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,NaN,NaN,NaN,10.068,8.089,62.0,/fMBookmwL6HjIgIVTjQ6EMr3pCH.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a...",65000000,84785914.0,Zodiac,19.083823,7.300,2080.0,/bgLyOROfFQI3FqYL7jQbiaV8lkN.jpg
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g...",NaN,NaN,NaN,3.744,10.000,1.0,/AizLGKf9hD2pXafKwMpBeJhdDq6.jpg
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,23600000,102391382.0,Zombieland,11.063029,7.200,3655.0,/vUzzDpVrab1BOG3ogxhRGfLN94d.jpg
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...",35000000,12506188.0,Zoom,11.967444,4.900,140.0,/so67uRr24dSQLt9YyZgBP5pNtc3.jpg


## Merging TV series budget data

`expensive_tv_series_with_budget.csv` contains per-season budget figures for a curated set of well-known series. Seasons are aggregated to series level (summing `budget_million` across seasons), converted to raw dollars, and merged on `title`. The resulting `budget_x`/`budget_y` conflict is resolved via `combine_first`.

Budget coverage ends up below 11% of all titles. `revenue` is just as sparse. Neither column was used in the final analysis; the cultural and geographic story we're telling doesn't depend on financial data this thin.

In [7]:
series_budget_aggregated = series_budgets_df.groupby('normalized_series', as_index=False)['budget_million'].sum()
series_budget_aggregated['budget'] = series_budget_aggregated['budget_million'] * 1e6
series_budget_aggregated.drop(columns=['budget_million'], inplace=True)
series_budget_aggregated.rename(columns={'normalized_series': 'title'}, inplace=True)

consolidated_df = consolidated_df.merge(
    series_budget_aggregated,
    on='title',
    how='left',
)
consolidated_df['budget'] = consolidated_df['budget_x'].combine_first(consolidated_df['budget_y'])
consolidated_df.drop(columns=['budget_x', 'budget_y'], inplace=True)
consolidated_df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,revenue,title_movie,popularity,vote_average,vote_count,poster_path,budget
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",NaN,NaN,14.193,7.900,285.0,/NFHy3z1taQ8nzudDHeVIJnih3J.jpg,NaN
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,NaN,NaN,30.808,7.090,133.0,/3E6IPkHH541ii41NKpU5loI4fcr.jpg,NaN
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",NaN,NaN,2.527,6.100,5.0,/pA7urHEBVDSJR7Cn7bANRbhwly7.jpg,NaN
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,NaN,NaN,10.068,8.089,62.0,/fMBookmwL6HjIgIVTjQ6EMr3pCH.jpg,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a...",84785914.0,Zodiac,19.083823,7.300,2080.0,/bgLyOROfFQI3FqYL7jQbiaV8lkN.jpg,65000000
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g...",NaN,NaN,3.744,10.000,1.0,/AizLGKf9hD2pXafKwMpBeJhdDq6.jpg,NaN
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,102391382.0,Zombieland,11.063029,7.200,3655.0,/vUzzDpVrab1BOG3ogxhRGfLN94d.jpg,23600000
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...",12506188.0,Zoom,11.967444,4.900,140.0,/so67uRr24dSQLt9YyZgBP5pNtc3.jpg,35000000


In [11]:
consolidated_df['date_added'] = pd.to_datetime(consolidated_df['date_added'].str.strip(), format='%B %d, %Y')
consolidated_df.date_added.describe()

count                          8797
mean     2019-05-17 05:59:08.436967
min             2008-01-01 00:00:00
25%             2018-04-06 00:00:00
50%             2019-07-02 00:00:00
75%             2020-08-19 00:00:00
max             2021-09-25 00:00:00
Name: date_added, dtype: object

## Parsing dates

`date_added` arrived as free-text strings in `"Month DD, YYYY"` format. These are parsed into datetime objects after stripping any leading or trailing whitespace from the source values.

The coverage issue shows up right away: only 138 titles (1.6%) have a `date_added` before 2016. Netflix existed well before that and had a catalog of 12,000+ titles as far back as 2008 ([2008 Annual Report, 10-K](https://s22.q4cdn.com/959853165/files/doc_financials/annual_reports/Final_AR_10K.pdf)). This dataset is a snapshot, captured at a single point in time. Titles that left the platform before the scrape date don't appear at all, and older titles that survived may have had their `date_added` overwritten or left blank. We treat the pre-2016 values as unreliable throughout the analysis.

In [8]:
consolidated_df.to_csv('./data/processed/consolidated_netflix_data.csv', index=False)

## Dataset overview

The consolidated dataset has **8,807 rows and 19 columns**. Each row is one title: a Movie or a TV Show in the Netflix catalog as of the scrape date.

**Composition:** 6,131 Movies (69.6%) and 2,676 TV Shows (30.4%). The 2:1 ratio reflects how Netflix counts its catalog by title rather than by watch-time or episode count. A 10-season series and a one-episode special each occupy exactly one row.

**Column coverage:**

| Column | Coverage | Notes |
|--------|----------|-------|
| `type`, `title`, `listed_in` | 100% | No gaps |
| `date_added` | ~99% | Pre-2016 values unreliable (1.6% of rows) |
| `rating` | ~99% | 4,435 movie rows had TV-style ratings; remapped to MPAA equivalents |
| `country` | ~93% | Multi-value field (comma-separated) |
| `duration` | ~99% | 3 rows had a duration string in the `rating` field, corrected |
| `vote_average`, `popularity` | 50.6% | 37.5% movies / 80.5% TV shows; perfectly correlated in coverage |
| `budget` | <11% | Sparse; excluded from analysis |
| `revenue` | <11% | Sparse; excluded from analysis |

Section 3 walks through each of these columns in detail and explains the cleaning decisions that followed.

# 3. Data Analysis
- Describe your data analysis and explain what you've learned about the dataset.

The phases below step through the consolidated dataset: coverage, composition, time, genre, content ratings, country, and TMDB community scores. Each phase shaped a decision about what the final figures should show, and what they should leave out.

In [17]:
#check if consolidated_df already exists to avoid re-running the whole merging process
if 'consolidated_df' not in locals():
    import pandas as pd
    consolidated_df = pd.read_csv('consolidated_netflix_data.csv')
total = len(consolidated_df)
movies_mask = consolidated_df['type'] == 'Movie'
tv_mask = consolidated_df['type'] == 'TV Show'

# Columns where numeric zero means "no data" — coerce to numeric first
numeric_nonzero = ['budget', 'revenue']
df_cov = consolidated_df.copy()
for col in numeric_nonzero:
    df_cov[col] = pd.to_numeric(df_cov[col], errors='coerce')

rows = []
for col in df_cov.columns:
    if col in numeric_nonzero:
        overall_nn = (df_cov[col].notna() & (df_cov[col] > 0)).sum()
        movies_nn  = (df_cov.loc[movies_mask, col].notna() & (df_cov.loc[movies_mask, col] > 0)).sum()
        tv_nn      = (df_cov.loc[tv_mask, col].notna()     & (df_cov.loc[tv_mask, col] > 0)).sum()
        note = ">0 only"
    else:
        overall_nn = df_cov[col].notna().sum()
        movies_nn  = df_cov.loc[movies_mask, col].notna().sum()
        tv_nn      = df_cov.loc[tv_mask, col].notna().sum()
        note = ""

    rows.append({
        'Column':   col,
        'Overall':  f"{overall_nn:,} / {total:,}  ({overall_nn/total:.1%})",
        'Movies':   f"{movies_nn:,} / {movies_mask.sum():,}  ({movies_nn/movies_mask.sum():.1%})",
        'TV Shows': f"{tv_nn:,} / {tv_mask.sum():,}  ({tv_nn/tv_mask.sum():.1%})",
        'Note':     note,
    })

coverage = pd.DataFrame(rows).set_index('Column')

def highlight_coverage(row):
    pct = float(row['Overall'].split('(')[1].rstrip('%)').strip()) / 100
    if pct < 0.20:
        return ['background-color: #FF0D2A'] * len(row)
    elif pct < 0.50:
        return ['background-color: #FF5809'] * len(row)
    return [''] * len(row)

coverage.style.apply(highlight_coverage, axis=1)

,Overall,Movies,TV Shows,Note
Column,,,,
show_id,"8,807 / 8,807 (100.0%)","6,131 / 6,131 (100.0%)","2,676 / 2,676 (100.0%)",
type,"8,807 / 8,807 (100.0%)","6,131 / 6,131 (100.0%)","2,676 / 2,676 (100.0%)",
title,"8,807 / 8,807 (100.0%)","6,131 / 6,131 (100.0%)","2,676 / 2,676 (100.0%)",
director,"6,173 / 8,807 (70.1%)","5,943 / 6,131 (96.9%)","230 / 2,676 (8.6%)",
cast,"7,982 / 8,807 (90.6%)","5,656 / 6,131 (92.3%)","2,326 / 2,676 (86.9%)",
country,"7,976 / 8,807 (90.6%)","5,691 / 6,131 (92.8%)","2,285 / 2,676 (85.4%)",
date_added,"8,797 / 8,807 (99.9%)","6,131 / 6,131 (100.0%)","2,666 / 2,676 (99.6%)",
release_year,"8,807 / 8,807 (100.0%)","6,131 / 6,131 (100.0%)","2,676 / 2,676 (100.0%)",
rating,"8,800 / 8,807 (99.9%)","6,126 / 6,131 (99.9%)","2,674 / 2,676 (99.9%)",


In [18]:
import plotly.graph_objects as go

type_counts = consolidated_df['type'].value_counts()

fig_type = go.Figure(go.Bar(
    x=type_counts.index.tolist(),
    y=type_counts.values.tolist(),
    text=[f"{v:,} ({v/len(consolidated_df):.1%})" for v in type_counts.values],
    textposition='outside',
    marker_color=['#E50914', '#564d4d'],
))

fig_type.update_layout(
    title=dict(text='<b>Dataset Composition: Movies vs. TV Shows</b>', x=0.5, xanchor='center', font=dict(color='white')),
    xaxis=dict(tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)'),
    yaxis=dict(title=dict(text='Count', font=dict(color='#cccccc')), tickfont=dict(color='#cccccc'),
               gridcolor='rgba(255,255,255,0.07)', range=[0, type_counts.max() * 1.15]),
    plot_bgcolor='#141414', paper_bgcolor='#141414',
    showlegend=False,
    margin=dict(t=70, b=50, l=60, r=40),
    width=500, height=400,
)
fig_type.show()

The dataset has roughly 6,100 Movies and 2,700 TV Shows, about a 2:1 ratio. This matters more than it might seem. Any count-based comparison between the two types will naturally favour movies just by volume, not because Netflix added movies more aggressively. Figure 1 shows movies being added at roughly twice the rate of TV shows per year, some of that gap is real, but some is just the base rates here.

TV shows are also undercounted in a different way: one row covers an entire series regardless of how many seasons or episodes it has. A long-running show and a one-season special count the same. That's worth keeping in mind when comparing the two types across any of the figures.

In [19]:
temporal = (
    consolidated_df.dropna(subset=['date_added'])
    .assign(year_added=lambda d: d['date_added'].dt.year)
    .groupby('year_added')
    .size()
    .reset_index(name='count')
)

pre2016  = temporal[temporal['year_added'] < 2016]['count'].sum()
post2016 = temporal[temporal['year_added'] >= 2016]['count'].sum()
print(f"Titles with date_added < 2016:  {pre2016:,}  ({pre2016/(pre2016+post2016):.1%})")
print(f"Titles with date_added >= 2016: {post2016:,}  ({post2016/(pre2016+post2016):.1%})")

fig_temporal = go.Figure(go.Bar(
    x=temporal['year_added'].tolist(),
    y=temporal['count'].tolist(),
    marker_color=['#555555' if y < 2016 else '#E50914' for y in temporal['year_added']],
    hovertemplate='<b>%{x}</b>: %{y} titles<extra></extra>',
))

fig_temporal.add_vline(x=2015.5, line_dash='dash', line_color='#FFD700', line_width=2)
fig_temporal.add_annotation(
    x=2015.5, y=temporal['count'].max() * 1.05,
    text="2016 reliability cutoff", showarrow=False,
    font=dict(color='#FFD700', size=11), xanchor='left', xshift=6,
)

fig_temporal.update_layout(
    title=dict(text='<b>Titles Added per Year (date_added)</b>', x=0.5, xanchor='center', font=dict(color='white')),
    xaxis=dict(title=dict(text='Year', font=dict(color='#cccccc')), tickmode='linear', dtick=1,
               tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)'),
    yaxis=dict(title=dict(text='Count', font=dict(color='#cccccc')), tickfont=dict(color='#cccccc'),
               gridcolor='rgba(255,255,255,0.07)'),
    plot_bgcolor='#141414', paper_bgcolor='#141414',
    margin=dict(t=70, b=60, l=60, r=40),
    width=800, height=420,
)
fig_temporal.show()

AttributeError: Can only use .dt accessor with datetimelike values

Only 138 titles (1.6%) have a `date_added` value before 2016. The other 98.4% are from 2016 onward. Netflix existed well before that, and had a large catalog by then.

This dataset is a scrape of the catalog at a single point in time, not a historical archive. Any title added years earlier but later removed from the platform simply wouldn't show up. Titles back-filled without their original add date would reflect whatever the scraper captured instead. Netflix's own [2008 Annual Report (10-K)](https://s22.q4cdn.com/959853165/files/doc_financials/annual_reports/Final_AR_10K.pdf) puts the streaming library at 12,000 titles by the end of that year, which makes the 138 pre-2016 entries look like just whatever happened to survive to scrape time with an intact date attached.

So we treat 2016 as where the data starts being reliable. The pre-2016 bars are still visible in Figure 1 for context, but the analysis doesn't lean on them.

In [20]:
# Explode multi-label listed_in column (comma-separated genres)
genres_exploded = (
    consolidated_df[['type', 'listed_in']]
    .dropna(subset=['listed_in'])
    .assign(genre=lambda d: d['listed_in'].str.split(', '))
    .explode('genre')
    .assign(genre=lambda d: d['genre'].str.strip())
)

# Map source genre labels to consolidated cross-type labels
genre_mapping = {
    'International Movies':      'International',
    'International TV Shows':    'International',
    'Dramas':                    'Drama',
    'TV Dramas':                 'Drama',
    'Comedies':                  'Comedy',
    'TV Comedies':               'Comedy',
    'Action & Adventure':        'Action & Adventure',
    'Documentaries':             'Documentary',
    'Docuseries':                'Documentary',
    'Children & Family Movies':  'Kids & Family',
    "Kids' TV":                  'Kids & Family',
    'Romantic Movies':           'Romance',
    'Romantic TV Shows':         'Romance',
    'Thrillers':                 'Thriller',
    'Crime TV Shows':            'Crime',
    'Horror Movies':             'Horror',
    'Stand-Up Comedy':           'Stand-Up Comedy',
    'Music & Musicals':          'Music & Musicals',
    'Reality TV':                'Reality TV',
    'Independent Movies':        'Independent Movies',
    'Anime Series':              'Anime',
    'British TV Shows':          'British TV',
}

genres_mapped = genres_exploded.copy()
genres_mapped['genre_consolidated'] = genres_mapped['genre'].map(genre_mapping)
genres_mapped = genres_mapped.dropna(subset=['genre_consolidated'])

# Compute % of each type's total for fair comparison
totals = genres_mapped.groupby('type').size()

genre_pct = (
    genres_mapped.groupby(['genre_consolidated', 'type'])
    .size()
    .reset_index(name='count')
    .assign(pct=lambda d: d.apply(lambda r: r['count'] / totals[r['type']] * 100, axis=1))
)

# Order by average pct across types
order = (
    genre_pct.groupby('genre_consolidated')['pct']
    .mean()
    .sort_values(ascending=True)
    .index.tolist()
)

fig_genre2 = go.Figure()
for content_type, color in [('Movie', '#E50914'), ('TV Show', '#564d4d')]:
    sub = genre_pct[genre_pct['type'] == content_type].set_index('genre_consolidated')
    fig_genre2.add_trace(go.Bar(
        name=content_type,
        y=order,
        x=[sub.loc[g, 'pct'] if g in sub.index else 0 for g in order],
        orientation='h',
        marker_color=color,
        hovertemplate=f'<b>{content_type}</b>: %{{x:.1f}}%<extra></extra>',
    ))

fig_genre2.update_layout(
    barmode='group',
    title=dict(
        text='<b>Consolidated Genre Share — Movies vs. TV Shows</b><br>'
             '<sup>% of each type\'s total tag appearances (genres merged across types)</sup>',
        x=0.5, xanchor='center', font=dict(color='white'),
    ),
    xaxis=dict(title=dict(text='% of type total', font=dict(color='#cccccc')),
               tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)',
               ticksuffix='%'),
    yaxis=dict(tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)'),
    plot_bgcolor='#141414', paper_bgcolor='#141414',
    legend=dict(font=dict(color='white'), bgcolor='rgba(30,30,30,0.8)'),
    margin=dict(t=90, b=60, l=180, r=40),
    width=850, height=620,
)
fig_genre2.write_image('./figures/genre_consolidated.png', scale=2)
fig_genre2.show()


The `listed_in` field is multi-label, so a single title can appear under several genre categories at once. Because Movies outnumber TV Shows roughly 2x, raw counts aren't a fair basis for comparison. The chart normalises each bar to a percentage of that type's total tag appearances, and merges parallel source labels ("Dramas" + "TV Dramas" into Drama, etc.) so the two types can be read side by side.

A few categories only appear on one side. "Crime TV Shows", "Horror Movies", "Thrillers", "Action & Adventure", "Reality TV", "British TV", and "Anime" each map to a consolidated bucket that has no equivalent label for the other type. That's a labeling artifact from the source data. Comparing those bars against zero is not very meaningful, so we only look at the shared genres for cross-type comparisons:

**International** is the biggest category on both sides, but TV Shows edge Movies here (around 27% vs 22%). International series make up a slightly higher slice of Netflix's TV catalog than international movies do of its film catalog. **Drama** leans toward Movies (roughly 20% vs 15%). Each TV series is one row regardless of how many seasons it has, so some of that gap may just be down to how drama content gets packaged in each format. **Comedy** is fairly close, Movies marginally ahead (around 14% vs 12%). **Documentary** is almost tied (about 7% for Movies vs 8% for TV), which makes sense given the source used separate labels ("Documentaries" for films, "Docuseries" for TV) that feed into the same bucket. **Kids & Family** tilts toward TV Shows (around 9% vs 5%), which fits the long-running episodic format most children's content comes in. **Romance** is slightly higher for TV Shows (around 7% vs 5%).

Genre coverage is clean. `listed_in` is 100% populated with no gaps. The range of shared genres is broad enough to support country-level breakdowns, which is what Figure 2 builds on.


In [21]:
from plotly.subplots import make_subplots

# 3 rows have duration strings ("66 min", "74 min", "84 min") in the rating field
duration_mask = consolidated_df['rating'].str.match(r'^\d+\s*min$', na=False)
n_duration = duration_mask.sum()
print(f"Duration-as-rating entries removed: {n_duration}")
consolidated_df.loc[duration_mask, 'rating'] = pd.NA

# normalise cross-system ratings for better comparison
# Netflix uses TV Parental Guidelines (TV-Y, TV-G, TV-PG, TV-14, TV-MA) as a fallback
# for films without an MPA rating, and occasionally MPAA labels appear on TV shows.
# Source: Wikipedia — TV Parental Guidelines
#   https://en.wikipedia.org/wiki/TV_Parental_Guidelines
# Source: MPA film rating system
#   https://www.motionpictures.org/film-ratings/

# TV Parental Guidelines → MPAA equivalents (applied to Movies)
tv_to_mpaa = {
    'TV-Y':     'G',      # ages 2+   ≈ G
    'TV-Y7':    'PG',     # ages 7+   ≈ PG
    'TV-Y7-FV': 'PG',     # ages 7+ (fantasy violence) ≈ PG
    'TV-G':     'G',      # general audiences ≈ G
    'TV-PG':    'PG',     # parental guidance ≈ PG
    'TV-14':    'PG-13',  # ages 14+  ≈ PG-13 (ages 13+)
    'TV-MA':    'R',      # mature/17+ ≈ R (17+)
}

# MPAA → TV Parental Guidelines equivalents (applied to TV Shows)
mpaa_to_tv = {
    'G':     'TV-G',
    'PG':    'TV-PG',
    'PG-13': 'TV-14',
    'R':     'TV-MA',
    'NC-17': 'TV-MA',
    'UR':    'NR',
}

movies_idx = consolidated_df['type'] == 'Movie'
tv_idx     = consolidated_df['type'] == 'TV Show'

rating_corrected = consolidated_df['rating'].copy()
rating_corrected[movies_idx] = rating_corrected[movies_idx].map(lambda r: tv_to_mpaa.get(r, r))
rating_corrected[tv_idx]     = rating_corrected[tv_idx].map(lambda r: mpaa_to_tv.get(r, r))
consolidated_df['rating_corrected'] = rating_corrected

n_movie_fixed = (consolidated_df.loc[movies_idx, 'rating'] != consolidated_df.loc[movies_idx, 'rating_corrected']).sum()
n_tv_fixed    = (consolidated_df.loc[tv_idx,     'rating'] != consolidated_df.loc[tv_idx,     'rating_corrected']).sum()
print(f"Movies with rating corrected:   {n_movie_fixed:,}")
print(f"TV Shows with rating corrected: {n_tv_fixed:,}")

# --- 3. Build ordered distributions ---
mpaa_order = ['G', 'PG', 'PG-13', 'R', 'NC-17', 'NR']
tv_order   = ['TV-Y', 'TV-Y7', 'TV-Y7-FV', 'TV-G', 'TV-PG', 'TV-14', 'TV-MA', 'NR']

movies_vc = consolidated_df[movies_idx].dropna(subset=['rating_corrected'])['rating_corrected'].value_counts()
tv_vc     = consolidated_df[tv_idx].dropna(subset=['rating_corrected'])['rating_corrected'].value_counts()

mpaa_present = [r for r in mpaa_order if movies_vc.get(r, 0) > 0]
tv_present   = [r for r in tv_order   if tv_vc.get(r, 0)     > 0]

print("\nMovies — final distribution:")
print(movies_vc[mpaa_present].to_string())
print("\nTV Shows — final distribution:")
print(tv_vc[tv_present].to_string())

# --- 4. Plot ---
fig_rating = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Movies (MPAA)', 'TV Shows (TV Parental Guidelines)'),
    horizontal_spacing=0.12,
)

fig_rating.add_trace(go.Bar(
    x=mpaa_present,
    y=[movies_vc.get(r, 0) for r in mpaa_present],
    marker_color='#E50914',
    name='Movie',
    hovertemplate='<b>%{x}</b>: %{y:,}<extra></extra>',
), row=1, col=1)

fig_rating.add_trace(go.Bar(
    x=tv_present,
    y=[tv_vc.get(r, 0) for r in tv_present],
    marker_color='#564d4d',
    name='TV Show',
    hovertemplate='<b>%{x}</b>: %{y:,}<extra></extra>',
), row=1, col=2)

fig_rating.update_layout(
    title=dict(
        text='<b>Content Rating Distribution — Movies vs. TV Shows</b><br>'
             '<sup>TV ratings mapped to MPAA for movies; MPAA ratings mapped to TV Parental Guidelines for TV shows</sup>',
        x=0.5, xanchor='center', font=dict(color='white'),
    ),
    plot_bgcolor='#141414', paper_bgcolor='#141414',
    showlegend=False,
    margin=dict(t=100, b=60, l=60, r=40),
    width=950, height=460,
)
fig_rating.update_xaxes(tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)')
fig_rating.update_yaxes(tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)')
fig_rating.update_yaxes(title_text='Count', title_font=dict(color='#cccccc'), row=1, col=1)
fig_rating.update_annotations(font=dict(color='white'))

fig_rating.write_image('./figures/rating_distribution.png', scale=2)
fig_rating.show()


Duration-as-rating entries removed: 0
Movies with rating corrected:   4,435
TV Shows with rating corrected: 4

Movies — final distribution:
rating_corrected
G         298
PG        971
PG-13    1917
R        2859
NC-17       3
NR         75

TV Shows — final distribution:
rating_corrected
TV-Y         176
TV-Y7        195
TV-Y7-FV       1
TV-G          94
TV-PG        323
TV-14        733
TV-MA       1147
NR             5


The `rating` column turned out to be a mix of two separate systems. US movies are rated by the MPA (G, PG, PG-13, R, NC-17), while TV shows use the TV Parental Guidelines (TV-Y, TV-G, TV-PG, TV-14, TV-MA), introduced in 1997 and since voluntarily adopted by streaming services including Netflix ([Wikipedia — TV Parental Guidelines](https://en.wikipedia.org/wiki/TV_Parental_Guidelines); [MPA film ratings](https://www.motionpictures.org/film-ratings/)). Netflix applies TV ratings to films that never received an official MPA certificate, and that affected 4,435 movies in this dataset. Only 4 TV show rows had MPAA labels and needed the reverse treatment. Mappings used: TV-MA → R, TV-14 → PG-13, TV-PG → PG, TV-G/TV-Y → G, TV-Y7/TV-Y7-FV → PG, and the reverse for TV shows (R → TV-MA, PG-13 → TV-14, PG → TV-PG, G → TV-G).

Three rows had duration strings ("66 min", "74 min", "84 min") sitting in the `rating` field, a fairly obvious data entry error. Those were set to NaN and dropped from the chart. Three rows out of 8,807 don't affect any downstream analysis, and Figure 3 is not impacted.

On the movie side, R is the biggest category at 2,859 titles (~47% of all movies). PG-13 is second at 1,917 (~31%). Those two adult-facing ratings together cover about 78% of the film catalog. G and PG combined come to around 21%. For TV shows, TV-MA leads at 1,147 titles (~44%), with TV-14 at 733 (~28%), putting about 72% of series in the two most mature categories. The children's tiers (TV-Y, TV-Y7, TV-G) make up roughly 18% of TV shows.

Both panels lean the same way: the catalog is predominantly adult content. Kids and family titles are present, though they take up a small share of the total. The same mature skew shows up again in Figure 3, where it gets sliced by time and genre.


In [22]:
country_exploded = (
    consolidated_df[['type', 'country']]
    .dropna(subset=['country'])
    .assign(country=lambda d: d['country'].str.split(', '))
    .explode('country')
    .assign(country=lambda d: d['country'].str.strip())
)

total_rows = len(consolidated_df)
rows_with_country = consolidated_df['country'].notna().sum()
coverage_pct = rows_with_country / total_rows
print(f"Rows with country data: {rows_with_country:,} / {total_rows:,}  ({coverage_pct:.1%})")

top20_countries = (
    country_exploded.groupby('country')
    .size()
    .sort_values(ascending=False)
    .head(20)
)

# For context: how many unique countries exist
n_unique = country_exploded['country'].nunique()
print(f"Unique countries in dataset: {n_unique}")
print("\nTop 20 countries (title appearances after explode):")
print(top20_countries.to_string())

top20_names = top20_countries.index.tolist()[::-1]  # reversed for horizontal bar

fig_country = go.Figure(go.Bar(
    y=top20_names,
    x=top20_countries[::-1].values.tolist(),
    orientation='h',
    marker_color='#E50914',
    hovertemplate='<b>%{y}</b>: %{x:,}<extra></extra>',
))

fig_country.update_layout(
    title=dict(
        text='<b>Top 20 Countries by Title Appearances</b><br>'
             '<sup>Country column exploded — one row per country tag</sup>',
        x=0.5, xanchor='center', font=dict(color='white'),
    ),
    xaxis=dict(title=dict(text='Title appearances', font=dict(color='#cccccc')),
               tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)'),
    yaxis=dict(tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)'),
    plot_bgcolor='#141414', paper_bgcolor='#141414',
    showlegend=False,
    margin=dict(t=90, b=60, l=160, r=40),
    width=800, height=580,
)

fig_country.write_image('./figures/country_distribution.png', scale=2)
fig_country.show()


Rows with country data: 7,976 / 8,807  (90.6%)
Unique countries in dataset: 127

Top 20 countries (title appearances after explode):
country
United States     3689
India             1046
United Kingdom     804
Canada             445
France             393
Japan              318
Spain              232
South Korea        231
Germany            226
Mexico             169
China              162
Australia          160
Egypt              117
Turkey             113
Hong Kong          105
Nigeria            103
Italy              100
Brazil              97
Argentina           91
Indonesia           90


Country data covers 90.6% of the catalog (7,976 out of 8,807 rows). The column is multi-value, so a co-production between the US and Canada counts for both. After exploding, the dataset spans 127 unique countries.

The US sits at 3,689 appearances, roughly 3.5x India in second place (1,046) and more than 4.5x the UK (804). That gap is wider than population or market size alone would explain. It mostly comes down to Netflix starting as a US domestic service with a catalog heavily weighted toward Hollywood production.

India and the UK are in third, before a noticeable drop to Canada (445) and France (393). Japan (318), Spain (232), South Korea (231), and Germany (226) follow in a tight cluster. From Mexico (169) down to Indonesia (90), fifteen countries all land between roughly 90 and 170 appearances, a band where the bars are almost the same length.

A few of those lower entries are worth pointing out. Egypt (117), Nigeria (103), and Indonesia (90) all make the top 20, which suggests real catalog presence from regions that typically get little representation in Western streaming data. South Korea at 231 is probably higher than expected given its population; Korean drama and film have had genuine international momentum over the past decade.

The long tail beyond the top 20 is steep. The 127 countries in total sounds broad, but most of them contribute only a handful of titles. Figure 2 builds on this concentration pattern.


In [23]:
import numpy as np
from plotly.subplots import make_subplots

va_col  = 'vote_average'
pop_col = 'popularity'

# Coerce both columns to numeric (handles any stray strings)
va_series  = pd.to_numeric(consolidated_df[va_col],  errors='coerce')
pop_series = pd.to_numeric(consolidated_df[pop_col], errors='coerce')

# Coverage by type
n_movies = movies_mask.sum()
n_tv     = tv_mask.sum()

va_movies  = va_series[movies_mask].notna().sum()
va_tv      = va_series[tv_mask].notna().sum()
pop_movies = pop_series[movies_mask].notna().sum()
pop_tv     = pop_series[tv_mask].notna().sum()

print("vote_average coverage:")
print(f"  Movies:   {va_movies:,} / {n_movies:,}  ({va_movies/n_movies:.1%})")
print(f"  TV Shows: {va_tv:,}  / {n_tv:,}   ({va_tv/n_tv:.1%})")
print(f"  Overall:  {va_series.notna().sum():,} / {len(consolidated_df):,}  ({va_series.notna().mean():.1%})")

print("\npopularity coverage:")
print(f"  Movies:   {pop_movies:,} / {n_movies:,}  ({pop_movies/n_movies:.1%})")
print(f"  TV Shows: {pop_tv:,}  / {n_tv:,}   ({pop_tv/n_tv:.1%})")

# Distributions for rows that have values
va_data_movies  = va_series[movies_mask  & va_series.notna()]
va_data_tv      = va_series[tv_mask      & va_series.notna()]
pop_data_movies = pop_series[movies_mask & pop_series.notna()]
pop_data_tv     = pop_series[tv_mask     & pop_series.notna()]

fig_va = make_subplots(
    rows=1, cols=2,
    subplot_titles=('vote_average distribution', 'popularity distribution (log₁₀ scale)'),
    horizontal_spacing=0.12,
)

# vote_average overlapping histograms
for data, name, color in [
    (va_data_movies, 'Movie',   '#E50914'),
    (va_data_tv,     'TV Show', '#564d4d'),
]:
    fig_va.add_trace(go.Histogram(
        x=data, name=name, marker_color=color,
        opacity=0.75, nbinsx=25,
        hovertemplate=f'<b>{name}</b><br>score: %{{x:.1f}}<br>count: %{{y}}<extra></extra>',
    ), row=1, col=1)

# popularity log-scale overlapping histograms
for data, name, color in [
    (pop_data_movies, 'Movie',   '#E50914'),
    (pop_data_tv,     'TV Show', '#564d4d'),
]:
    log_data = np.log10(data.clip(lower=0.01))
    fig_va.add_trace(go.Histogram(
        x=log_data, name=name, marker_color=color,
        opacity=0.75, nbinsx=30, showlegend=False,
        hovertemplate=f'<b>{name}</b><br>log10(popularity): %{{x:.2f}}<br>count: %{{y}}<extra></extra>',
    ), row=1, col=2)

fig_va.update_layout(
    title=dict(
        text='<b>vote_average & popularity — rows with values only</b><br>'
             '<sup>Coverage: Movies ~37.5%, TV Shows ~80.5%</sup>',
        x=0.5, xanchor='center', font=dict(color='white'),
    ),
    barmode='overlay',
    plot_bgcolor='#141414', paper_bgcolor='#141414',
    legend=dict(font=dict(color='white'), bgcolor='rgba(30,30,30,0.8)'),
    margin=dict(t=90, b=60, l=70, r=40),
    width=950, height=440,
)
fig_va.update_xaxes(tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)')
fig_va.update_yaxes(tickfont=dict(color='#cccccc'), gridcolor='rgba(255,255,255,0.07)')
fig_va.update_yaxes(title_text='Count', title_font=dict(color='#cccccc'), row=1, col=1)
fig_va.update_xaxes(title_text='vote_average (0–10)', title_font=dict(color='#cccccc'), row=1, col=1)
fig_va.update_xaxes(title_text='log₁₀(popularity)', title_font=dict(color='#cccccc'), row=1, col=2)
fig_va.update_annotations(font=dict(color='white'))

fig_va.write_image('./figures/vote_popularity_coverage.png', scale=2)
fig_va.show()


vote_average coverage:
  Movies:   2,301 / 6,131  (37.5%)
  TV Shows: 2,155  / 2,676   (80.5%)
  Overall:  4,456 / 8,807  (50.6%)

popularity coverage:
  Movies:   2,301 / 6,131  (37.5%)
  TV Shows: 2,155  / 2,676   (80.5%)


Both `vote_average` and `popularity` come from The Movie Database (TMDB), a community-driven platform where registered users rate and track titles. The dataset here was assembled by scraping the TMDB API, so coverage depends on whether a title had a TMDB entry at the time of collection.

**`vote_average`** is the mean of all user-submitted ratings on a 0–10 scale. These are crowdsourced votes from the TMDB community, not critic scores or editorial assessments. The rating for any given title is just the arithmetic mean of however many individual ratings have been submitted to TMDB at that point.<sup>[[1]](https://www.themoviedb.org)</sup>

**`popularity`** is more involved. TMDB recalculates it daily for every title using a mix of signals: votes cast that day, page views that day, favourites added, watchlist additions, the title's release date, total vote count, and the previous day's score. For TV shows the calculation also factors in when the next and last episodes aired. The score has no fixed ceiling and it decays over time. A film that gets a burst of attention after a re-release will see a spike, then gradual decay. Because the raw values span several orders of magnitude (a niche title might score 1–5 while a blockbuster peaks in the thousands), the histograms here use a log₁₀ axis.<sup>[[2]](https://developer.themoviedb.org/docs/popularity-and-trending)</sup>

Coverage for both columns is 50.6% overall, with a clear split: 37.5% of movies have values (2,301 out of 6,131) versus 80.5% of TV shows (2,155 out of 2,676). The two columns share identical coverage as they both come from the same complimentary dataset.

The thin movie sample is worth noting. TMDB tends to have better coverage for titles that circulated internationally or picked up an online audience. Smaller regional films are less likely to have a TMDB entry at all, so the 2,301 movies with scores probably skew toward titles that were already reasonably well-known outside their home country. Any comparison using these values needs to treat the movie side as a biased sample.

The `vote_average` distribution peaks in the 5.5–7 range for both types, with a right-skewed tail trailing off past 8. There's a spike at 0 for movies (around 140 entries) and a smaller one for TV shows. These are almost certainly placeholder zeros in the source data. A title averaging 0/10 from real users would be unusual enough to remember. The `popularity` histograms peak around log₁₀ ≈ 1 (popularity ≈ 10) for both types, but TV shows have a wider, flatter distribution that extends further into high values. Long-running series accumulate daily engagement signals across many seasons and years, which pushes their lifetime scores higher under TMDB's formula.

With 4,456 rows that have scores, aggregates by genre or country are stable enough to report. Comparisons at the individual title level, particularly on the movie side where 62.5% of the catalog has no score, would not hold up. Figure 4 uses genre-level averages for this reason.



### Analytical Decisions

The sections above turned up a handful of data issues that had real consequences for the figures. Rather than bury those decisions in code comments, they're collected here.

**Budget and revenue were dropped.** Both columns have below 11% non-null values after the metadata merge. That's not enough to draw any useful pattern from, so the financial angle in the original specification got dropped. The figures focus instead on growth, geography, content ratings, and audience reception.

**Pre-2016 `date_added` data is treated as unreliable.** The dataset was collected at a single point in time. That matters because Netflix removes titles over time. If a film was added in 2013 but removed before the scrape happened, it won't show up at all. The gap in pre-2016 additions almost certainly reflects this rather than actual catalogue history. Figure 1 still plots the full timeline, but the interpretation there avoids reading anything into the early years.

**The rating column was corrected before use.** Netflix sometimes applies TV Parental Guidelines ratings to films that lack an MPA rating. In the raw data, 4,435 movies carried TV ratings (TV-MA, TV-14, TV-PG, etc.) and 4 TV shows carried MPA ratings. Each was remapped to the closest equivalent: TV-MA → R, TV-14 → PG-13, TV-PG → PG, TV-G / TV-Y → G, TV-Y7 → PG.

**`vote_average` and `popularity` are used only for aggregates.** Movie coverage is 37.5%, and those 2,301 titles lean toward internationally known films. Scores for individual titles aren't reliable given the sample size and the coverage bias. Figure 4 compares at the genre level, where the averages hold up.

These four decisions changed the direction of the project. The original specification pointed toward financial analysis; the final figures ended up being about cultural patterns instead. That's a better fit for what the data can actually support.


# 4. Genre. Which genre of data story did you use?
- Which tools did you use from each of the 3 categories of Visual Narrative (Figure 7 in Segal and Heer). Why?
- Which tools did you use from each of the 3 categories of Narrative Structure (Figure 7 in Segal and Heer). Why?


### Style
The story follows a magazine-style format: a linear guided narrative that moves through sections, each combining interactive charts with our reflections. This mirrors how a data journalism piece works.

### Visual Narrative

**Visual Structuring**
We used several of the tools Segel & Heer describe here: a consistent visual platform, establishing shots, and progress bars. Every figure shares the same Netflix-branded theme, background, and color convention, so readers don't have to re-learn the visual language between charts. Section numbers 1 through 8 serve as a progress marker, and the opening motivation section acts as an establishing shot that frames the story before any data appears.

**Transition Guidance**
Object continuity is used here. The color coding for Movies and TV Shows stays mostly identical across all six figures, so readers can carry meaning from one chart to the next without reorientation.

### Narrative Structure

**Ordering**
The story is linear. Sections are numbered 1 through 8, and each builds on what came before. The temporal growth figure comes before genre and geography because understanding when Netflix expanded is context for understanding what it expanded into.

**Interactivity**
Every chart supports hover highlighting. Readers can hover over individual bars to see exact counts, or hover over milestone markers to read the associated event.

**Messaging**
Each figure is preceded by a reflection section that sets up the question being answered and follows with an analysis block that interprets what the chart shows. Section headings act as clear signposts. Figure 1 carries its messaging directly on the chart through annotated milestone markers, which is the most visible example of in-chart messaging in the piece.

# 5. Visualizations
- Explain the visualizations you've chosen.
- Why are they right for the story you want to tell?


### Figure 1 — The Explosion
**Question:** How did Netflix's catalog grow over the years?

#### Visualization Choice
A **stacked bar chart** showing titles added per year, split by Movie vs. TV Show, with **golden milestone markers** at key moments in Netflix's history.

**Why a stacked bar chart?**
It shows two things at once: total catalog growth (bar height) and the shifting balance between Movies and TV Shows (fill ratio). A line chart captures the trend, but you'd lose the composition story. A plain count of all titles combined would hide the Movie/TV split entirely, which becomes more relevant later in the story.

**Why annotation pins?**
Netflix's growth isn't a smooth curve. It jumps at specific moments tied to real decisions: international launches, the debut of original content, major platform pivots. Pinning these events directly on the chart turns a simple count into something more readable. Instead of wondering why things spiked around 2016, you can see it was the 130-territory expansion. Hover tooltips keep the visual clean while still making the full detail available.

#### Why This Figure Is Right for the Story
The story we're telling is about how Netflix grew from an American movie service into a global content platform. This figure opens that story by showing *when* the growth happened and *why*. It's the most author-guided part of the piece: no exploration required, just context. The geography, content strategy, and audience resonance figures that follow need this timeline as a baseline.

One note on coverage: `date_added` and `type` are populated for nearly every row, so sampling bias isn't a concern at the data level. But completeness is a different question, and it matters here.

#### Analysis & Data Limitations

> **Worth noting: this figure shows how many titles were added each year, not how large Netflix's total catalog was at any given time.**

The near-zero bars before 2016 look suspicious, and they probably are. In 2010, Netflix expanded internationally to Canada, yet the dataset shows only a handful of titles added that year. Netflix's own [2008 Annual Report (10-K)](https://s22.q4cdn.com/959853165/files/doc_financials/annual_reports/Final_AR_10K.pdf) puts the streaming library at 12,000 titles by the end of that year, which makes the single-digit counts in the early chart years hard to take at face value. The most likely explanation is that titles from the pre-2016 era weren't back-filled with their original `date_added` values, or were later pulled from the catalog and dropped from the dataset entirely.

So we treat 2016 as where the reliable data starts. From there, the trend holds together and lines up with what we'd expect from Netflix's known history.

**What the post-2016 data shows:**
- Movies and TV Shows follow roughly the same shape: growth from 2016 to 2019, then a pullback through 2021. The peak and decline likely reflects the COVID-19 production slowdown, plus Netflix becoming more selective as competition increased.
- Movies consistently outnumber TV Shows by more than 2:1. At the 2019 peak, over 1,400 movies were added in a single year; for TV Shows that number was around 595.
- The gap isn't surprising. Each TV series counts as one row no matter how many episodes it has, while every film gets its own entry. The production math also just favors more movies than series in raw title count.

In [24]:
import plotly.graph_objects as go

# aggregate data by year and type
fig1_df = consolidated_df.dropna(subset=['date_added']).copy()
fig1_df['year_added'] = fig1_df['date_added'].dt.year

yearly = (
    fig1_df.groupby(['year_added', 'type'])
    .size()
    .reset_index(name='count')
)

all_years  = sorted(yearly['year_added'].unique())
movies_y   = yearly[yearly['type'] == 'Movie'].set_index('year_added')['count']
tv_y       = yearly[yearly['type'] == 'TV Show'].set_index('year_added')['count']
max_total  = yearly.groupby('year_added')['count'].sum().max()

# hand picked milestones based on Netflix history and major industry events
milestones = [
    (2009, "2009", "Netflix Originals label launched"),
    (2010, "2010", "International expansion begins (Canada)"),
    (2011, "2011", "Latin America expansion (Argentina, Chile, Mexico…)"),
    (2012, "2012", "European expansion (UK, Ireland, Nordics)"),
    (2013, "2013", "House of Cards — first Netflix Original"),
    (2015, "2015", "Japan expansion"),
    (2016, "2016", "Global expansion to 130 new territories"),
    (2018, "2018", "Netflix created Netflix Animation, its first production studio."),
    (2021, "2021", "Netflix launches its gaming platform Netflix Games"),
]


fig = go.Figure()

fig.add_trace(go.Bar(
    x=all_years,
    y=[movies_y.get(y, 0) for y in all_years],
    name='Movie',
    marker_color='#E50914',
    hovertemplate='<b>Movies:</b> %{y}<extra></extra>',
))

fig.add_trace(go.Bar(
    x=all_years,
    y=[tv_y.get(y, 0) for y in all_years],
    name='TV Show',
    marker_color='#564d4d',
    hovertemplate='<b>TV Shows:</b> %{y}<extra></extra>',
))

# Milestone markers — alternating heights so labels don't collide
flag_ys = [max_total * (1.04 + 0.06 * (i % 2)) for i in range(len(milestones))]

fig.add_trace(go.Scatter(
    x=[m[0] for m in milestones],
    y=flag_ys,
    mode='markers',
    marker=dict(
        symbol='triangle-down', size=14,
        color='#FFD700', line=dict(width=1.5, color='#B8860B'),
    ),
    name='Key Milestone',
    hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}<extra></extra>',
    customdata=[(m[1], m[2]) for m in milestones],
))

# Vertical dotted lines
for year, _, _ in milestones:
    fig.add_vline(x=year, line_dash='dot',
                  line_color='rgba(255,215,0,0.35)', line_width=1.5)

# ── 4. Layout ─────────────────────────────────────────────────────────────────
fig.update_layout(
    barmode='stack',
    title=dict(
        text=(
            '<b>The Explosion: Netflix Catalog Growth Over Time</b><br>'
            '<sup>Titles added per year — hover ▼ markers for key milestones</sup>'
        ),
        font=dict(size=18, color='white'),
        x=0.5, xanchor='center',
    ),
    xaxis=dict(
        title=dict(text='Year Added to Netflix', font=dict(color='#cccccc')),
        tickmode='linear', tick0=2008, dtick=1,
        tickfont=dict(color='#cccccc'),
        gridcolor='rgba(255,255,255,0.07)',
    ),
    yaxis=dict(
        title=dict(text='Titles Added', font=dict(color='#cccccc')),
        tickfont=dict(color='#cccccc'),
        gridcolor='rgba(255,255,255,0.07)',
        range=[0, max_total * 1.18],
    ),
    plot_bgcolor='#141414',
    paper_bgcolor='#141414',
    legend=dict(
        font=dict(color='white'), bgcolor='rgba(30,30,30,0.8)',
        bordercolor='rgba(255,255,255,0.2)', borderwidth=1,
    ),
    hovermode='x unified',
    margin=dict(t=110, b=60, l=70, r=40),
    width=950, height=560,
)

fig.write_html('./figures/figure1_explosion.html')
fig.write_image('./figures/figure1_explosion.png', scale=2)
fig.show()

AttributeError: Can only use .dt accessor with datetimelike values

### Figure 2  -  What does each country specialize in? ###
### Question
Which countries specialize in which genres, and how does that differ between Movies and TV Shows?

### Data used
- **Source columns:** `country`, `type`, `listed_in` (raw Netflix genre tags), `title`
- **Coverage:** `country` is present for ~91% of rows; `type` and `listed_in` are present for ~100%
- **Country handling:** when a row lists multiple co-production countries (e.g. `"United States, United Kingdom"`), only the *first* country is used. This keeps each title attributed to a single primary producer and avoids double-counting.
- **Filtering:** rows are kept only if the country maps to a known ISO-3 code (the lookup table in the code covers ~115 countries) and `type` is either `Movie` or `TV Show`.

### How the genre signal is built
Netflix's `listed_in` field is a free-form string with multiple labels per title (e.g. `"International TV Shows, TV Dramas, TV Mysteries"`). To make countries comparable, every label is collapsed into one of **10 canonical genre buckets** by keyword matching:

| Bucket | Keywords matched (case-insensitive, substring) |
|---|---|
| Drama | `drama` |
| Comedy | `comed`, `stand-up`, `talk` |
| Action | `action`, `adventure` |
| Thriller | `thriller`, `suspense` |
| Crime | `crime` |
| Romance | `romantic`, `romance` |
| Documentary | `documentar`, `docuseries`, `nature`, `reality`, `travel` |
| Animation | `animat`, `anime`, `kids`, `children` |
| Horror | `horror` |
| Sci Fi | `sci-fi`, `sci fi`, `science fiction`, `fantasy` |

A single title can contribute to multiple buckets (e.g. an `"International Movies, Romantic Movies, Comedies"` title counts toward Romance *and* Comedy). The per-country score for each genre is the count of titles whose `listed_in` matched at least one keyword for that bucket, kept separately for Movies and TV Shows.

### From scores to "estimated quantity"
The side panel shows an **estimated quantity** rather than a raw count. The reason: a country's titles can match multiple genre buckets at once, so summing the raw genre counts double-counts. To produce numbers that *do* sum to roughly the country's total, the score for each genre is normalised against the sum of all genre scores for that country/type, then multiplied by the country's actual title count:

```
quantity(genre) = round( score(genre) / sum(scores) * total_titles_of_that_type )
```

This gives a clean "X out of Y titles look like Drama" reading per country, while keeping the genres ranked in the same order the raw scores would have produced.

### Findings
The visible patterns when clicking through countries:

- **United States** dominates volume in every genre but skews toward Drama, Comedy, and Documentary — consistent with it being the catalog's volume backbone.
- **India** is heavily Drama- and Comedy-led on the Movie side, with a much smaller TV Show footprint, reflecting Netflix India's earlier reliance on licensed Bollywood films before original series ramped up.
- **South Korea** has a strikingly different profile between Movies and TV Shows: K-dramas dominate its TV catalog, while its Movie slice leans more on Thriller and Crime.
- **Japan** is the clearest outlier on Animation — its Movies and especially its TV Shows index disproportionately on Animation/Anime, whereas no other major producer matches that share.
- **United Kingdom** looks similar to the US but with a noticeably stronger Crime and Documentary lean, especially on TV.
- **Spain, Mexico, Brazil, Turkey** all show telenovela-style profiles: Drama and Romance lead, with Crime and Thriller close behind on the TV side.

The Movies/TV toggle is the most informative interaction: most countries' fingerprints are *very* different across the two formats. The Movie panel for a country is often Drama- and Comedy-heavy because that is what Netflix licenses in bulk; the TV panel is where a country's *original-production* identity shows up most clearly (K-drama, anime, Spanish-language thrillers, British crime).

### Caveats
- Keyword matching is coarse. A title labelled `"Stand-Up Comedy"` lands in Comedy correctly, but Netflix's own taxonomy has dozens of subgenres, and rare ones (Faith & Spirituality, LGBTQ Movies, Sports Movies) are intentionally not surfaced as top-level buckets here.
- The 9% of rows without a country are dropped from this figure entirely, which slightly under-counts globally produced or anonymously credited titles.

In [25]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Match the dark Netflix-style Plotly theme
NETFLIX_BG = 'rgb(20,20,20)'
NETFLIX_PANEL = 'rgb(31,31,31)'
NETFLIX_GRID = 'rgba(255,255,255,0.07)'
NETFLIX_TEXT = 'rgb(245,245,245)'
NETFLIX_MUTED = 'rgb(204,204,204)'
NETFLIX_RED = 'rgba(227,0,11,0.647)'
NETFLIX_DARK_RED = 'rgb(178,7,16)'
NETFLIX_GRAY = 'rgb(86,77,77)'
NETFLIX_GOLD = 'rgb(255,215,0)'
NETFLIX_BORDER = 'rgba(255,255,255,0.2)'
NETFLIX_HOVER = 'rgba(30,30,30,0.92)'
NETFLIX_REDSCALE = [
    [0.00, 'rgb(223,64,74)'],
    [0.25, 'rgb(219,56,56)'],
    [0.60, 'rgb(225,34,34)'],
    [1.00, 'rgba(227,0,11,0.647)'],]

In [26]:
#with the help of an LLM we made a mapping of country names to ISO-3 codes, 
# which we use to create a new 'iso3' column for easier grouping and visualization 
# by country. We also define genre keywords to classify the 'listed_in' genres into
#  broader categories, creating 'matched_genres' and 'main_genre' columns for
#  more flexible analysis of genre trends.
COUNTRY_ISO3 = {
    'Afghanistan':'AFG','Albania':'ALB','Algeria':'DZA','Angola':'AGO','Argentina':'ARG','Armenia':'ARM',
    'Australia':'AUS','Austria':'AUT','Azerbaijan':'AZE','Bahrain':'BHR','Bangladesh':'BGD','Belarus':'BLR',
    'Belgium':'BEL','Bolivia':'BOL','Bosnia and Herzegovina':'BIH','Brazil':'BRA','Bulgaria':'BGR',
    'Burkina Faso':'BFA','Cambodia':'KHM','Cameroon':'CMR','Canada':'CAN','Chile':'CHL','China':'CHN',
    'Colombia':'COL','Costa Rica':'CRI','Croatia':'HRV','Cuba':'CUB','Cyprus':'CYP','Czech Republic':'CZE',
    'Denmark':'DNK','Dominican Republic':'DOM','Ecuador':'ECU','Egypt':'EGY','El Salvador':'SLV',
    'Estonia':'EST','Ethiopia':'ETH','Finland':'FIN','France':'FRA','Georgia':'GEO','Germany':'DEU',
    'Ghana':'GHA','Greece':'GRC','Guatemala':'GTM','Honduras':'HND','Hong Kong':'HKG','Hungary':'HUN',
    'Iceland':'ISL','India':'IND','Indonesia':'IDN','Iran':'IRN','Iraq':'IRQ','Ireland':'IRL',
    'Israel':'ISR','Italy':'ITA','Jamaica':'JAM','Japan':'JPN','Jordan':'JOR','Kazakhstan':'KAZ',
    'Kenya':'KEN','Kuwait':'KWT','Latvia':'LVA','Lebanon':'LBN','Libya':'LBY','Lithuania':'LTU',
    'Luxembourg':'LUX','Malaysia':'MYS','Malta':'MLT','Mexico':'MEX','Montenegro':'MNE','Morocco':'MAR',
    'Mozambique':'MOZ','Myanmar':'MMR','Nepal':'NPL','Netherlands':'NLD','New Zealand':'NZL',
    'Nicaragua':'NIC','Nigeria':'NGA','Norway':'NOR','Oman':'OMN','Pakistan':'PAK','Panama':'PAN',
    'Paraguay':'PRY','Peru':'PER','Philippines':'PHL','Poland':'POL','Portugal':'PRT','Qatar':'QAT',
    'Romania':'ROU','Russia':'RUS','Rwanda':'RWA','Saudi Arabia':'SAU','Senegal':'SEN','Serbia':'SRB',
    'Singapore':'SGP','Slovakia':'SVK','Slovenia':'SVN','Somalia':'SOM','South Africa':'ZAF',
    'South Korea':'KOR','Spain':'ESP','Sri Lanka':'LKA','Sudan':'SDN','Sweden':'SWE','Switzerland':'CHE',
    'Taiwan':'TWN','Tanzania':'TZA','Thailand':'THA','Tunisia':'TUN','Turkey':'TUR','Uganda':'UGA',
    'Ukraine':'UKR','United Arab Emirates':'ARE','United Kingdom':'GBR','United States':'USA',
    'Uruguay':'URY','Venezuela':'VEN','Vietnam':'VNM','Yemen':'YEM','Zambia':'ZMB','Zimbabwe':'ZWE'
}

GENRE_KEYWORDS = {
    'Drama': ['drama'],
    'Comedy': ['comed', 'stand-up', 'talk'],
    'Action': ['action', 'adventure'],
    'Thriller': ['thriller', 'suspense'],
    'Crime': ['crime'],
    'Romance': ['romantic', 'romance'],
    'Documentary': ['documentar', 'docuseries', 'nature', 'reality', 'travel'],
    'Animation': ['animat', 'anime', 'kids', 'children'],
    'Horror': ['horror'],
    'Sci Fi': ['sci-fi', 'sci fi', 'science fiction', 'fantasy'],
}
GENRES = list(GENRE_KEYWORDS)

def primary_country(country_text):
    if pd.isna(country_text) or str(country_text).strip() == '':
        return np.nan
    return str(country_text).split(',')[0].strip()

def matched_genres(listed_in):
    text = str(listed_in or '').lower()
    return [genre for genre, keys in GENRE_KEYWORDS.items() if any(key in text for key in keys)]

def first_matched_genre(listed_in):
    matches = matched_genres(listed_in)
    return matches[0] if matches else 'Other'

figs24_df = consolidated_df.copy()
figs24_df['primary_country'] = figs24_df['country'].apply(primary_country)
figs24_df['iso3'] = figs24_df['primary_country'].map(COUNTRY_ISO3)
figs24_df['matched_genres'] = figs24_df['listed_in'].apply(matched_genres)
figs24_df['main_genre'] = figs24_df['listed_in'].apply(first_matched_genre)

print('Rows with country:', figs24_df['country'].notna().sum())
print('Rows with known ISO-3 country:', figs24_df['iso3'].notna().sum())
print(figs24_df['type'].value_counts())

Rows with country: 7976
Rows with known ISO-3 country: 7967
type
Movie      6131
TV Show    2676
Name: count, dtype: int64


In [27]:
fig2_rows = figs24_df[
    figs24_df['type'].isin(['Movie', 'TV Show'])
    & figs24_df['primary_country'].notna()
    & figs24_df['iso3'].notna()
].copy()

long_genres = fig2_rows.explode('matched_genres')
long_genres = long_genres[long_genres['matched_genres'].notna()].copy()

# Raw genre scores: one title can contribute to multiple genre buckets.
genre_scores = (
    long_genres
    .groupby(['primary_country', 'iso3', 'type', 'matched_genres'])
    .size()
    .rename('score')
    .reset_index()
)

totals = (
    fig2_rows
    .groupby(['primary_country', 'iso3', 'type'])
    .size()
    .rename('total_titles')
    .reset_index()
)

score_sum = (
    genre_scores
    .groupby(['primary_country', 'type'])['score']
    .sum()
    .rename('score_sum')
    .reset_index()
)

fig2_summary = (
    genre_scores
    .merge(totals, on=['primary_country', 'iso3', 'type'], how='left')
    .merge(score_sum, on=['primary_country', 'type'], how='left')
)
fig2_summary['estimated_quantity'] = np.round(
    fig2_summary['score'] / fig2_summary['score_sum'] * fig2_summary['total_titles']
).astype(int)

def country_hover_table(content_type):
    subset = fig2_summary[fig2_summary['type'] == content_type].copy()
    top = (
        subset.sort_values(['primary_country', 'estimated_quantity'], ascending=[True, False])
        .groupby(['primary_country', 'iso3', 'total_titles'])
        .head(5)
    )
    top_text = (
        top.assign(item=lambda x: x['matched_genres'] + ': ' + x['estimated_quantity'].astype(str))
        .groupby(['primary_country', 'iso3', 'total_titles'])['item']
        .apply(lambda items: '<br>'.join(items))
        .reset_index(name='top_genres')
    )
    return top_text.sort_values('total_titles', ascending=False)

fig2_movie = country_hover_table('Movie')
fig2_tv = country_hover_table('TV Show')

def make_map_trace(table, name, visible):
    return go.Choropleth(
        locations=table['iso3'],
        z=table['total_titles'],
        text=table['primary_country'],
        customdata=np.stack([table['total_titles'], table['top_genres']], axis=1),
        colorscale='reds',
        marker_line_color=NETFLIX_BG,
        marker_line_width=0.7,
        colorbar=dict(
            title=dict(text='Titles', font=dict(color=NETFLIX_TEXT)),
            tickfont=dict(color=NETFLIX_MUTED),
            bgcolor=NETFLIX_PANEL,
            bordercolor=NETFLIX_BORDER,
            borderwidth=1,
        ),
        visible=visible,
        name=name,
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Titles: %{customdata[0]}<br><br>'
            '<b>Top genre quantities</b><br>%{customdata[1]}'
            '<extra></extra>'
        ),
    )

fig2 = go.Figure([
    make_map_trace(fig2_movie, 'Movies', True),
    make_map_trace(fig2_tv, 'TV Shows', False),
])

fig2.update_layout(
    geo=dict(
        projection_type='natural earth',
        bgcolor=NETFLIX_BG,
        showland=True,
        landcolor=NETFLIX_PANEL,
        showcountries=True,
        countrycolor='rgba(255,255,255,0.18)',
        showocean=True,
        oceancolor=NETFLIX_BG,
        coastlinecolor='rgba(255,255,255,0.25)',
        showframe=False,
    ),
    margin=dict(l=20, r=20, t=70, b=10),
    height=620,
    paper_bgcolor=NETFLIX_BG,
    plot_bgcolor=NETFLIX_BG,
    font=dict(color=NETFLIX_MUTED),
    hoverlabel=dict(
        bgcolor=NETFLIX_HOVER,
        bordercolor=NETFLIX_BORDER,
        font=dict(color=NETFLIX_TEXT),
    ),
    updatemenus=[
        dict(
            type='buttons',
            direction='right',
            x=0.01,
            y=1.08,
            bgcolor=NETFLIX_RED,
            bordercolor=NETFLIX_BORDER,
            font=dict(color=NETFLIX_TEXT),
            showactive=False,
            buttons=[dict(
                label='Movies',
                method='update',
                args=[
                    {'visible': [True, False]},
                    {'updatemenus[0].bgcolor': NETFLIX_RED, 'updatemenus[1].bgcolor': NETFLIX_PANEL},
                ],
            )],
        ),
        dict(
            type='buttons',
            direction='right',
            x=0.15,
            y=1.08,
            bgcolor=NETFLIX_PANEL,
            bordercolor=NETFLIX_BORDER,
            font=dict(color=NETFLIX_TEXT),
            showactive=False,
            buttons=[dict(
                label='TV Shows',
                method='update',
                args=[
                    {'visible': [False, True]},
                    {'updatemenus[0].bgcolor': NETFLIX_PANEL, 'updatemenus[1].bgcolor': NETFLIX_RED},
                ],
            )],
        ),
    ],
)

fig2.show()

### Figure 3 - The mature content bet

**Question that we want to answer:** How did Netflix's content profile shift over time? did it increasingly bet on mature, adult-oriented content?

Content ratings can be a signal of audience targeting. A platform that increases its bet in R-rated titles is making a choice to serve adult viewers more, one that grows its Kids and Family share is building a different type of audience. Looking at its rating composition over time we can see in which direction Netflix has been going.

The analysis covers **2016–2021**. Data before 2016 comes from a single catalog scrape, which means older titles that were later removed from Netflix are not reflected, pre 2016 numbers dont represent what was actually added those years, so we only look at the period 2016-2021.
This stacked area chart shows the share of each audience tier per year, with a toggle to inspect **Movies** or **TV Shows** separately. Hover to compare rating shares across all tiers for a given year.

### Caveats
- **Ratings are self-assigned.** Netflix rates the tittles itself, not by an independent body. The MPAA and TV Parental Guidelines provide a framework, but  Netflix originals may have ratings that reflect a marketing strategy. Netflix increased its original production significantly after 2016. Because originals are rated at the platforms discretion and tend to target adult subscribers, they may inflate the Mature share.
- **The chart shows *what*, not *why*.** The data can establish that the composition changed, but it cannot explain whether the shift was driven by subscribers, competitive pressure from other streamers, content cost, or a strategy.

### Analysis

**TV Shows:** The platform begins with a more balanced mix of categories tiers, but the mature category grows noticeably over the period. Kids and Family programming decline as a share of new TV shows, while Mature titles increasingly dominate the total percentage. The shift is gradua, suggesting a sustained strategy.

**Movies:** Netflix films were centered around mature audiences from the start. The notable change within Movies is the rise of the Teens tier (PG-13, TV-14), which grows from around 18% in 2016 to nearly 35% by 2021. 

**Across both:** the declining share of Kids and Family programming is consistent in both charts, suggesting Netflix shifting its focus from all audiences content in favour of programming for adults.

In [13]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

Netflix's raw catalog uses two parallel rating systems: the **MPAA system** (G, PG, PG-13, R, NC-17) for Movies and the **TV Parental Guidelines** (TV-Y, TV-Y7, TV-PG, TV-14, TV-MA) for TV Shows. To be able to compare the two formats on the same chart, we merge the two systems in 5 general categories:

| Tier | Ratings |
|------|---------|
| Kids | G, TV-Y, TV-Y7, TV-Y7-FV |
| Family | PG, TV-PG |
| Teens | PG-13, TV-14 |
| Mature | R, TV-MA, NC-17 |
| Unrated | NR, UR, or anything unrecognised |

Some rows in the  data CSV file have duration strings (`"74 min"`, `"84 min"`) in the `rating` column, so we stripped them before mapping.

The `date_added` column is parsed as a date, rows where parsing fails are then dropped, the result is filtered to 2016–2021.

In [14]:
DATA_FILE = Path("consolidated_netflix_data.csv")

RATING_GROUPS = {
    "Kids":    {"ratings": {"G", "TV-Y", "TV-Y7", "TV-Y7-FV"}, "label": "Kids (G, TV-Y, TV-Y7)",         "color": "#4ade80"},
    "Family":  {"ratings": {"PG", "TV-PG"},                     "label": "Family (PG, TV-PG)",             "color": "#fbbf24"},
    "Teens":   {"ratings": {"PG-13", "TV-14"},                  "label": "Teens (PG-13, TV-14)",           "color": "#f97316"},
    "Mature":  {"ratings": {"R", "TV-MA", "NC-17"},             "label": "Mature (R, TV-MA, NC-17)",       "color": "#be123c"},
    "Unrated": {"ratings": {"NR", "UR"},                        "label": "Unrated (NR, UR, Other)",        "color": "#cbd5e1"},
}
GROUP_ORDER = ["Kids", "Family", "Teens", "Mature", "Unrated"]

BAD_RATINGS = {"74 min", "84 min", "66 min", ""}


def load_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_FILE)
    df["date_added"] = pd.to_datetime(df["date_added"], format="%B %d, %Y", errors="coerce")
    df["year"] = df["date_added"].dt.year
    df = df[df["year"].notna()].copy()
    df["year"] = df["year"].astype(int)
    df = df[(df["year"] >= 2016) & (df["year"] <= 2021)]

    df["rating"] = df["rating"].fillna("").str.strip()
    df = df[~df["rating"].isin(BAD_RATINGS)]

    rating_to_group = {}
    for group, info in RATING_GROUPS.items():
        for r in info["ratings"]:
            rating_to_group[r] = group

    df["group"] = df["rating"].map(rating_to_group).fillna("Unrated")
    return df[["year", "group", "type"]].reset_index(drop=True)


df = load_data()
print(f"{len(df):,} rows after filtering (2016–2021)")
df.head()

8,575 rows after filtering (2016–2021)


,year,group,type
0,2021,Teens,Movie
1,2021,Mature,TV Show
2,2021,Mature,TV Show
3,2021,Mature,TV Show
4,2021,Mature,TV Show


The data is grouped by year and audience tier, then  counts are converted to **percentages** so every year sums to 100%. Using shares rather than raw counts is important because Netflix added more titles in 2020–21 than in 2016, so absolute numbers would make later years look dominant simply because of catalog growth. Percentage share solves that problem.

In [15]:
def compute_shares(df: pd.DataFrame, content_type: str):
    subset = df[df["type"] == content_type]
    counts = (
        subset.groupby(["year", "group"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=GROUP_ORDER, fill_value=0)
    )
    pct = counts.div(counts.sum(axis=1), axis=0) * 100
    return pct, counts


movies_pct, movies_cnt = compute_shares(df, "Movie")
print("Movies — % share per year:")
movies_pct.round(1)

Movies — % share per year:


group,Kids,Family,Teens,Mature,Unrated
year,,,,,
2016,8.0,11.2,18.3,51.0,11.6
2017,5.3,14.6,25.6,50.2,4.3
2018,4.0,12.9,32.7,47.3,3.1
2019,3.8,14.4,32.8,47.0,2.0
2020,5.9,13.9,32.8,44.4,3.0
2021,6.5,11.7,34.8,44.9,2.0


We chosed to use a stacked area chart because it shows two things, the total % share and how each tier share rises or falls relative to the others.

The y-axis shows percentage share, not title counts. Each content type, Movies or TV Shows, gets its own set of five traces.

Hovering activates a tooltip that shows all five tiers' shares for that year at once, making it easier to compare how the composition shifted without clicking through each individually.

In [16]:
def build_figure(df: pd.DataFrame) -> go.Figure:
    views = {"Movies": "Movie", "TV Shows": "TV Show"}
    view_labels = list(views.keys())
    n_groups = len(GROUP_ORDER)

    traces = []
    for v_idx, (label, content_type) in enumerate(views.items()):
        pct_df, count_df = compute_shares(df, content_type)
        years = sorted(pct_df.index.tolist())
        is_visible = v_idx == 0

        for group in GROUP_ORDER:
            info = RATING_GROUPS[group]
            pct_vals = [pct_df.loc[y, group] if y in pct_df.index else 0.0 for y in years]
            count_vals = [int(count_df.loc[y, group]) if y in count_df.index else 0 for y in years]
            hover = [
                f"<b>{info['label']}</b><br>Year: {y}<br>Share: {p:.1f}%<br>Titles: {c}"
                for y, p, c in zip(years, pct_vals, count_vals)
            ]
            traces.append(go.Scatter(
                x=years,
                y=pct_vals,
                name=info["label"],
                stackgroup="one",
                fillcolor=info["color"],
                line=dict(width=0.5, color=info["color"]),
                mode="lines",
                visible=is_visible,
                text=hover,
                hoverinfo="text",
                showlegend=True,
                legendgroup=group,
                legendgrouptitle_text=None,
            ))

    def vis_array(active_view_idx: int) -> list:
        return [(i // n_groups) == active_view_idx for i in range(len(traces))]

    buttons = [
        dict(
            label=label,
            method="update",
            args=[{"visible": vis_array(i)}, {}],
        )
        for i, label in enumerate(view_labels)
    ]

    fig = go.Figure(data=traces)
    fig.update_layout(
        xaxis=dict(
            title="Year added to Netflix",
            tickmode="linear",
            dtick=1,
            title_font=dict(color='#cccccc'),
            tickfont=dict(color='#cccccc'),
            gridcolor='rgba(255,255,255,0.07)',
        ),
        yaxis=dict(
            title="Share of titles (%)",
            range=[0, 100],
            title_font=dict(color='#cccccc'),
            tickfont=dict(color='#cccccc'),
            gridcolor='rgba(255,255,255,0.07)',
        ),
        hovermode="x unified",
        plot_bgcolor='#141414',
        paper_bgcolor='#141414',
        legend=dict(
            orientation="h",
            y=1.05,
            x=0,
            xanchor="left",
            yanchor="bottom",
            traceorder="normal",
            font=dict(color='white'),
        ),
        width=900,
        height=600,
        margin=dict(l=60, r=40, t=150, b=60),
        updatemenus=[dict(
            type="buttons",
            direction="right",
            x=0.0,
            xanchor="left",
            y=1.33,
            buttons=buttons,
            showactive=True,
            bgcolor="white",
            bordercolor="#cccccc",
            font=dict(color="black"),
        )],
    )
    return fig


fig = build_figure(df)
fig.show()

### Figure 4 — What resonates by genre and country?
### Question
Which genres and countries produce the most popular and highest-rated content on Netflix?

#### Data used
- **Source columns:** raw `vote_average` and `popularity` from TMDB, plus `listed_in`, `country`, and `type`
- **Derived columns:** `log_vote_average = log1p(vote_average)` and `log_popularity = log1p(popularity)`
- **Coverage:** `vote_average` and `popularity` are populated for ~50% of the catalog (about 4,400 titles): roughly **80% of TV Shows** matched against TMDB, but only **~38% of Movies**. This is the largest data caveat for the figure.
- **Filtering:** a row is kept only if both `vote_average` and `popularity` parse to finite numbers. Rows missing either are silently excluded from the chart.

### How the metrics are computed
- **Log vote average:** TMDB `vote_average` is originally a user-rating score on a 0–10 scale. For the visualization, Figure 4 uses `log1p(vote_average)` so the x-axis and the popularity axis are both shown on transformed scales. This makes the scatter easier to read visually, but it also compresses differences among already-high ratings.
- **Log popularity:** TMDB popularity is a daily, time-decaying score that combines page views, votes, favourites, watchlist activity, release timing, and trending signals on the TMDB site. It is *not* a viewership number. The raw score is extremely right-skewed, with a few breakout titles far above the rest, so Figure 4 uses `log1p(popularity)` instead of raw popularity. This compresses the long tail and makes ordinary titles visible without letting a few viral hits dominate the scale.
- **Per-genre / per-country averages** in the bottom chart are simple unweighted means of `log1p(vote_average)` and `log1p(popularity)` across the titles in that group, after filtering by `type` (Movies or TV Shows). These averages should be read as relative comparisons, not as direct audience-size or quality estimates.

#### Two views
1. **Top scatter — Log rating vs log popularity:** every dot is one title, coloured by genre or by country. The legend shows the top 10 groups by title count; hovering or clicking a legend entry isolates that group. Additionally, in order to visually compare similar scales between average voting and popularity properly.
2. **Bottom bar+line — Average log rating and average log popularity by group:** for the chosen breakdown (genre or country), bars show average log rating and the line shows average log popularity on a secondary axis. The list is sorted by average log rating, descending.

#### A correction to how countries are ranked
The original implementation sorted countries purely by average rating with no minimum sample size. That surfaced countries like **Mozambique** (1 title) and **Nepal** (1 title) at the top of the list — a single high-rated entry was enough to leapfrog volume producers. That is statistically meaningless and visually misleading.

The fix applied: when the breakdown is **Country**, only countries with **≥ 20 titles in the current Movie/TV filter** are eligible for the ranking. Genre rankings are unchanged because every genre bucket already contains hundreds of titles. The threshold of 20 is a practical compromise — it strips out one-off matches without being so strict that small but real markets (Belgium, Argentina, Sweden) disappear.

#### Findings

**By genre (TV Shows):** Anime, Documentary, and Crime tend to score highest on average log rating, while broad Drama and Comedy buckets sit slightly lower. One interpretation is that niche genres attract more self-selecting audiences on TMDB, so their ratings can be more forgiving. Average log popularity is led by broad, high-visibility genres such as Action/Adventure and Drama, where Netflix has many globally marketed shows.

**By genre (Movies):** the spread between genres is narrower. Documentaries punch above their weight on average log rating, while broad International Movies and Dramas sit closer to the centre because those buckets include a long tail of licensed catalog titles. Because both metrics are log-transformed, the movie view emphasizes relative differences without letting a few extreme popularity values dominate the plot.

**By country (with the new threshold):**
- The top countries by *average log rating* are now markets like **South Korea, Japan, the United Kingdom, and Taiwan** — all of which combine reasonable catalog volume with strong TMDB reception.
- The **United States** sits closer to the middle on average log rating despite producing the most titles. This is less a simple quality judgment than a volume effect: a very large catalog includes both hits and a long tail of ordinary titles.
- **India and Mexico** show relatively strong average log popularity but more moderate average log ratings. That suggests broad audience attention in TMDB terms, but not necessarily higher viewer satisfaction or higher Netflix viewership.

**Scatter view:** for both Movies and TV Shows, the cloud is easier to compare because both axes use `log1p(...)`. The transformation keeps breakout titles visible while reducing their ability to flatten the rest of the catalog into the bottom of the plot.

#### Caveats
- **Coverage bias:** with only ~38% of Movies matched against TMDB, the Movie view is noticeably less representative than the TV view (~80%). TMDB tends to have better metadata for English-language and internationally visible content, so smaller or less globally circulated Movie producers are under-represented in this figure.
- **Popularity is not viewership.** Even after log transformation, it reflects TMDB-site attention at the time the snapshot was taken. Titles that trended after the snapshot — or titles with devoted Netflix audiences but little TMDB activity — are systematically undervalued.
- **Log transforms change interpretation.** Differences on both axes are not raw additive gaps. For popularity especially, a one-unit difference in `log1p(popularity)` corresponds to a multiplicative difference in the original TMDB score.
- **Log rating compresses the high end.** The transform makes the visual scales more comparable, but it also makes the difference between, for example, 8 and 9 look smaller than on the raw 0–10 rating scale. The plot is best used for pattern comparison, not precise score interpretation.
- **Vote average flattens recency.** A 2-year-old hit and a 15-year-old classic with similar ratings are treated identically here, even though their cultural footprints are very different.


In [32]:
fig4_rows = figs24_df.copy()
fig4_rows['vote_average'] = pd.to_numeric(fig4_rows['vote_average'], errors='coerce')
fig4_rows['popularity'] = pd.to_numeric(fig4_rows['popularity'], errors='coerce')

# Log-transform both plotted metrics so the visual comparison uses comparable transformed scales.
fig4_rows['log_vote_average'] = fig4_rows['vote_average'].apply(lambda x: np.log1p(x))
fig4_rows['log_popularity'] = fig4_rows['popularity'].apply(lambda x: np.log1p(x))
fig4_rows = fig4_rows[
    fig4_rows['title'].notna()
    & fig4_rows['type'].isin(['Movie', 'TV Show'])
    & fig4_rows['log_vote_average'].notna()
    & fig4_rows['log_popularity'].notna()
    & fig4_rows['primary_country'].notna()
].copy()

print(f'Rows with both vote_average and popularity: {len(fig4_rows):,}')
print(fig4_rows.groupby('type').size())
fig4_rows[['title', 'type', 'primary_country', 'main_genre', 'vote_average', 'log_vote_average', 'popularity', 'log_popularity']].head()


Rows with both vote_average and popularity: 4,143
type
Movie      2254
TV Show    1889
dtype: int64


,title,type,primary_country,main_genre,vote_average,log_vote_average,popularity,log_popularity
1,Blood & Water,TV Show,South Africa,Drama,7.900,2.186051,14.193000,2.720835
4,Kota Factory,TV Show,India,Comedy,8.089,2.207065,10.068000,2.404058
7,Sankofa,Movie,United States,Drama,6.900,2.066863,0.045860,0.044840
15,Dear White People,TV Show,United States,Drama,6.200,1.974081,5.546847,1.878984
21,Resurrection: Ertugrul,TV Show,Turkey,Drama,7.600,2.151762,24.467000,3.237383


In [33]:
PALETTE = [NETFLIX_RED, NETFLIX_GRAY, NETFLIX_GOLD, '#8b0000', '#f87171', '#a3a3a3',
           '#f97316', '#b91c1c', '#facc15', '#737373', '#ef4444', '#d4d4d4']


def view_label(content_type, breakdown):
    content_label = 'Movies' if content_type == 'Movie' else 'TV Shows'
    return f'{content_label} by {breakdown}'


def scatter_traces(content_type, breakdown):
    subset = fig4_rows[fig4_rows['type'] == content_type].copy()
    key_col = 'main_genre' if breakdown == 'genre' else 'primary_country'
    top_keys = subset[key_col].value_counts().head(10).index.tolist()
    traces = []
    for i, key in enumerate(top_keys):
        pts = subset[subset[key_col] == key]
        traces.append(go.Scatter(
            x=pts['log_vote_average'],
            y=pts['log_popularity'],
            mode='markers',
            name=str(key),
            legendgroup=str(key),
            text=pts['title'],
            customdata=np.stack([pts['primary_country'], pts['main_genre'], pts['type']], axis=1),
            marker=dict(
                size=8,
                opacity=0.72,
                color=PALETTE[i % len(PALETTE)],
                line=dict(color=NETFLIX_BG, width=0.5),
            ),
            hovertemplate=(
                '<b>%{text}</b><br>'
                'Log rating: %{x:.2f}<br>'
                'Log Popularity: %{y:.2f}<br>'
                'Country: %{customdata[0]}<br>'
                'Genre: %{customdata[1]}<br>'
                'Type: %{customdata[2]}'
                '<extra></extra>'
            ),
        ))
    return traces


def summary_metrics(content_type, breakdown):
    subset = fig4_rows[fig4_rows['type'] == content_type].copy()
    key_col = 'main_genre' if breakdown == 'genre' else 'primary_country'
    metrics = (
        subset.groupby(key_col)
        .agg(
            count=('title', 'size'),
            avg_log_rating=('log_vote_average', 'mean'),
            avg_log_popularity=('log_popularity', 'mean'),
        )
        .reset_index()
        .rename(columns={key_col: 'group'})
    )
    if breakdown == 'country':
        metrics = metrics[metrics['count'] >= 20]
    return metrics.sort_values('avg_log_rating', ascending=False).head(12)


def summary_traces(content_type, breakdown):
    metrics = summary_metrics(content_type, breakdown)
    return [
        go.Bar(
            x=metrics['group'],
            y=metrics['avg_log_rating'],
            name='Avg log rating',
            marker_color=NETFLIX_RED,
            showlegend=False,
            customdata=np.stack([metrics['count'], metrics['avg_log_popularity']], axis=1),
            hovertemplate='%{x}<br>Avg log rating: %{y:.2f}<br>Titles: %{customdata[0]}<br>Avg log popularity: %{customdata[1]:.2f}<extra></extra>',
        ),
        go.Scatter(
            x=metrics['group'],
            y=metrics['avg_log_popularity'],
            mode='lines+markers',
            name='Avg log popularity',
            marker=dict(color=NETFLIX_GOLD, size=8),
            line=dict(color=NETFLIX_GOLD, width=2),
            showlegend=False,
            hovertemplate='%{x}<br>Avg log popularity: %{y:.2f}<extra></extra>',
        ),
    ]


views = []
for content_type in ['Movie', 'TV Show']:
    for breakdown in ['genre', 'country']:
        scatter = scatter_traces(content_type, breakdown)
        summary = summary_traces(content_type, breakdown)
        views.append({
            'label': view_label(content_type, breakdown),
            'content_type': content_type,
            'breakdown': breakdown,
            'scatter_count': len(scatter),
            'legend_title': 'Top genres' if breakdown == 'genre' else 'Top countries',
            'summary_x': 'Genre' if breakdown == 'genre' else 'Country',
            'traces': scatter + summary,
        })

all_traces = []
trace_ranges = []
trace_positions = []
for view in views:
    start = len(all_traces)
    for local_idx, trace in enumerate(view['traces']):
        trace.visible = False
        all_traces.append(trace)
        if local_idx < view['scatter_count']:
            trace_positions.append((1, False))
        elif local_idx == view['scatter_count']:
            trace_positions.append((2, False))
        else:
            trace_positions.append((2, True))
    trace_ranges.append((start, len(all_traces)))

for trace in all_traces[trace_ranges[0][0]:trace_ranges[0][1]]:
    trace.visible = True

fig4 = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.35,
    row_heights=[0.58, 0.42],
    specs=[[{}], [{'secondary_y': True}]],
)

for trace, (row, secondary_y) in zip(all_traces, trace_positions):
    fig4.add_trace(trace, row=row, col=1, secondary_y=secondary_y)

N_VIEWS = len(views)
x_positions = [0.00, 0.15, 0.30, 0.45]
updatemenu_dicts = []

for i, (view, (start, end)) in enumerate(zip(views, trace_ranges)):
    visible = [False] * len(all_traces)
    for idx in range(start, end):
        visible[idx] = True

  
    bgcolor_updates = {
        f'updatemenus[{j}].bgcolor': (NETFLIX_RED if j == i else NETFLIX_PANEL)
        for j in range(N_VIEWS)
    }

    label = 'Genre' if view['breakdown'] == 'genre' else 'Country'

    updatemenu_dicts.append(dict(
        type='buttons',
        direction='right',
        x=x_positions[i],
        y=1.22,
        xanchor='left',
        yanchor='top',
        showactive=False,
        bgcolor=NETFLIX_RED if i == 0 else NETFLIX_PANEL,
        bordercolor=NETFLIX_BORDER,
        font=dict(color=NETFLIX_TEXT),
        pad=dict(t=0, r=4),
        buttons=[dict(
            label=label,
            method='update',
            args=[
                {'visible': visible},
                {
                    'legend': {'title': {'text': view['legend_title']}},
                    'xaxis2': {'title': '', 'tickangle': -25},
                    **bgcolor_updates,
                },
            ],
        )],
    ))

fig4.update_layout(
    height=930,
    margin=dict(l=70, r=90, t=190, b=105),
    plot_bgcolor=NETFLIX_BG,
    paper_bgcolor=NETFLIX_BG,
    font=dict(color=NETFLIX_MUTED),
    hovermode='closest',
    hoverlabel=dict(
        bgcolor=NETFLIX_HOVER,
        bordercolor=NETFLIX_BORDER,
        font=dict(color=NETFLIX_TEXT),
    ),
    legend=dict(
        title=dict(text=views[0]['legend_title'], font=dict(color=NETFLIX_TEXT)),
        font=dict(color=NETFLIX_TEXT),
        bgcolor='rgba(30,30,30,0.8)',
        bordercolor=NETFLIX_BORDER,
        borderwidth=1,
        orientation='h',
        y=1.005,
        x=0,
        xanchor='left',
        yanchor='bottom',
        itemwidth=50,
        tracegroupgap=4,
    ),
    updatemenus=updatemenu_dicts,
    annotations=[
        dict(text='Movies', x=0.00, y=1.265, xref='paper', yref='paper', showarrow=False, xanchor='left', font=dict(size=13, color=NETFLIX_MUTED)),
        dict(text='TV Shows', x=0.25, y=1.265, xref='paper', yref='paper', showarrow=False, xanchor='left', font=dict(size=13, color=NETFLIX_MUTED)),
        dict(text='Log rating vs log popularity', x=0.00, y=1.3, xref='paper', yref='paper', showarrow=False, xanchor='left', font=dict(size=15, color=NETFLIX_TEXT)),
        dict(text='Sorted average log rating and log popularity', x=0.00, y=0.395, xref='paper', yref='paper', showarrow=False, xanchor='left', font=dict(size=15, color=NETFLIX_TEXT)),
    ],
)

fig4.update_xaxes(title_text='Log vote average', title_standoff=30, tickangle=-35, row=1, col=1)
fig4.update_yaxes(title_text='Log Popularity', row=1, col=1)


fig4.update_xaxes(title_text='', tickangle=-25, row=2, col=1)
fig4.update_yaxes(title_text='Average log rating', range=[0, np.log1p(10)], row=2, col=1, secondary_y=False)
fig4.update_yaxes(title_text='Average  log popularity', showgrid=False, row=2, col=1, secondary_y=True)

fig4.update_xaxes(
    tickfont=dict(color=NETFLIX_MUTED),
    title_font=dict(color=NETFLIX_MUTED),
    gridcolor=NETFLIX_GRID,
    zerolinecolor=NETFLIX_GRID,
    linecolor=NETFLIX_BORDER,
)
fig4.update_yaxes(
    tickfont=dict(color=NETFLIX_MUTED),
    title_font=dict(color=NETFLIX_MUTED),
    gridcolor=NETFLIX_GRID,
    zerolinecolor=NETFLIX_GRID,
    linecolor=NETFLIX_BORDER,
)

fig4.show()


In [34]:
OUTPUT_DIR = Path('./figures')
fig2_html = OUTPUT_DIR / 'figure2_country_genre_map.html'
fig4_html = OUTPUT_DIR / 'figure4_rating_popularity.html'

fig2.write_html(fig2_html, include_plotlyjs='cdn', full_html=True)
fig4.write_html(fig4_html, include_plotlyjs='cdn', full_html=True)


print(f'Saved interactive Figure 2 HTML to {fig2_html}')
print(f'Saved interactive Figure 4 HTML to {fig4_html}')

try:
    import kaleido  # noqa: F401
    fig2_png = OUTPUT_DIR / 'figure2_country_genre_map.png'
    fig4_png = OUTPUT_DIR / 'figure4_rating_popularity.png'
    fig2.write_image(fig2_png, width=1200, height=760, scale=2)
    fig4.write_image(fig4_png, width=1200, height=760, scale=2)
    print(f'Saved Figure 2 PNG to {fig2_png}')
    print(f'Saved Figure 4 PNG to {fig4_png}')
except Exception as exc:
    print('PNG export skipped. Install kaleido to enable fig.write_image(...).')
    print(f'Reason: {type(exc).__name__}: {exc}')


Saved interactive Figure 2 HTML to figures\figure2_country_genre_map.html
Saved interactive Figure 4 HTML to figures\figure4_rating_popularity.html
Saved Figure 2 PNG to figures\figure2_country_genre_map.png
Saved Figure 4 PNG to figures\figure4_rating_popularity.png


# 6. Discussion. Think critically about your creation
### What went well?
- Data preprocessing went quite smoothly, given we needed to merge data from plenty of different sources.
- In general, communication between team members was effiecient. Everyone's opinion was heard and everyone contributed to other peoples responbilities by giving ideas or exposing flaws. That happened througha a course of consistent team meetings (1-2 times per week), as well as constant communication from whats App messages.
- Managing to actually produce some reasonable results in the end
- Workload division was split fairly through all members. No one needed to do other peoples work for them.
### What is still missing? What could be improved?, Why?
- The dates of the released movies and tv shows seemed to have missing points before 2015 that is due to the dataset we used. We could maybe have done a deeper research to find the data, but this would be to time consuming and maybe we wouldn't be able to finish on time.
- There were some uncertainty on whether the choice of plots was proper in order to explain our data storyline (ex. there is some uncertainty for the use of figure 4 was the correct way to perform comparison between average voting and popularity). Maybe we could have used more time to discuss more thoroughly over the selection of plots.


# 7. Contributions. Who did what?
#### Lucas Patricio Martin Campopiano (s250484)
- Design of the outline of all figures shown
- Code and Analysis of Figure 3
#### Levente Murgás (s242957)
- Website development
- Dataset merging and preprocessing
- Code and Analysis of Figure 1
#### Konstantinos Papadopoulos (s250219)
- Code and Analysis of Figure 2
- Code and Analysis of Figure 4
#### Criteria
The criteria we used for spliting workload depended on the amount of cumulative workload was necessary for each person, because we wanted to distribute everything fairly. Additionally, it is self-eplenatory, but every member is aware and has contributed of others tasks.

# 8. References

Netflix's 2008 Annual Report (10-K): https://s22.q4cdn.com/959853165/files/doc_financials/annual_reports/Final_AR_10K.pdf

The Movie Database (TMDB). User ratings. https://www.themoviedb.org*

TMDB Developer Documentation. Popularity & Trending. https://developer.themoviedb.org/docs/popularity-and-trending*
